# Cleaning Notebook

This notebook shows the data cleaning workflow used to transform raw SMS exports into a consistent dataset for analysis and modeling.

The notebook covers:
- loading raw SMS export data
- anonymizing sensitive fields in message content and phone data
- tagging failed, balance, airtime, and adjustment messages
- filtering out non-behavioral OTP and promotional text
- removing duplicates and invalid records
- generating a unique `transaction_id` per message for tracking
- saving the cleaned dataset and a data quality report

**Demographics note:** Demographic enrichment (`age`, `gender`, `location`, `profession`) is handled downstream during feature engineering, not in this cleaning notebook.

**Instructions:** Run the cells in order. Each cell includes comments explaining what it does and what output to expect.

In [12]:
# Cell 1: Import required libraries
# This cell loads the libraries needed for data processing and text matching.
# Run this first to set up the environment.

from pathlib import Path
import pandas as pd
import re
import uuid

print("Libraries imported successfully.")

Libraries imported successfully.


In [13]:
# Cell 2: Set up file paths
# This cell defines the paths to input and output files relative to the notebook location.
# Expected output: Confirmation of paths.

base_dir = Path.cwd().parent
raw_path = base_dir / '1_Data_Collection' / 'raw_data.csv'
cleaned_path = base_dir / '2_Data_Cleaning' / 'cleaned_data.csv'
quality_path = base_dir / '2_Data_Cleaning' / 'data_quality_report.csv'

print(f"Raw data path: {raw_path}")
print(f"Cleaned data will be saved to: {cleaned_path}")
print(f"Quality report will be saved to: {quality_path}")

Raw data path: d:\one drivee\OneDrive\Desktop\Final data sc\project\submission\DataScience_Final_GroupA_TeamGamma\1_Data_Collection\raw_data.csv
Cleaned data will be saved to: d:\one drivee\OneDrive\Desktop\Final data sc\project\submission\DataScience_Final_GroupA_TeamGamma\2_Data_Cleaning\cleaned_data.csv
Quality report will be saved to: d:\one drivee\OneDrive\Desktop\Final data sc\project\submission\DataScience_Final_GroupA_TeamGamma\2_Data_Cleaning\data_quality_report.csv


In [14]:
# Cell 3: Load raw data
# This cell reads the raw SMS export data and ensures text columns are properly formatted.
# Expected output: Number of rows and columns loaded.

raw = pd.read_csv(raw_path)
print(f'Loaded raw data with {len(raw):,} rows and {len(raw.columns)} columns.')

# Ensure text columns are treated consistently.
raw['content'] = raw['content'].astype(str)

# Generate a stable transaction_id for each message.
raw = raw.reset_index(drop=True)
def build_transaction_id(row_index: int, source_file: str | None) -> str:
    source = source_file if source_file else "unknown_source"
    key = f"{source}|{row_index}"
    return f"txn-{uuid.uuid5(uuid.NAMESPACE_DNS, key)}"

source_values = raw['source_file'] if 'source_file' in raw.columns else pd.Series([None] * len(raw))
raw['transaction_id'] = [build_transaction_id(idx, source) for idx, source in enumerate(source_values.tolist())]

Loaded raw data with 25,996 rows and 13 columns.


In [15]:
# Cell 3.5: Anonymize sensitive information
# This cell applies the same anonymization approach used in src/cleaning.py.
# Expected output: Preview of anonymized content and phone values.


# Patterns aligned with production cleaner
name_re = re.compile(r"\b[A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)+\b")
phone_re = re.compile(r"\b\d{8,12}\b")
transaction_id_re = re.compile(
    r"\b(?:transaction\s*id|identifiant\s+de\s+transaction|id\s+de\s+transaction|financial\s+transaction\s+id|external\s+transaction\s+id)\b[:\s-]*([A-Za-z0-9-]+)",
    re.IGNORECASE,
 )
email_re = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b")

def anonymize_content(text: str) -> str:
    """Anonymize personal information in SMS content."""
    content = str(text)
    content = transaction_id_re.sub("ID_MASKED", content)
    content = phone_re.sub("XXXX", content)
    content = name_re.sub("Mr. X", content)
    content = email_re.sub("[EMAIL]", content)
    return content

raw['content'] = raw['content'].apply(anonymize_content)
if 'phone' in raw.columns:
    raw['phone'] = raw['phone'].apply(lambda text: phone_re.sub("XXXX", "" if pd.isna(text) else str(text)))

print("Sensitive information anonymized in content and phone.")
print("Anonymization patterns applied:")
print("- Transaction identifiers replaced with ID_MASKED")
print("- Phone numbers replaced with XXXX")
print("- Email addresses replaced with [EMAIL]")
print("- Proper names replaced with Mr. X")

preview_cols = ['content'] + (['phone'] if 'phone' in raw.columns else [])
display(raw[preview_cols].head())

Sensitive information anonymized in content and phone.
Anonymization patterns applied:
- Transaction identifiers replaced with ID_MASKED
- Phone numbers replaced with XXXX
- Email addresses replaced with [EMAIL]
- Proper names replaced with Mr. X


,content,phone
0,Mr. X: 5032 FCFA ; Solde disponible: 5032 FCFA...,MobileMoney
1,Transfert de 2500 FCFA effectue avec succes a ...,MobileMoney
2,Mr. X: 1977 FCFA ; Solde disponible: 1977 FCFA...,MobileMoney
3,Une transaction de 1000 XAF effectue par MTNC ...,MobileMoney
4,Une transaction de 250 XAF effectue par MTNC B...,MobileMoney


In [16]:
# Cell 4: Define regex patterns for message categorization
# This cell sets up regular expressions to identify different types of SMS messages.
# Expected output: List of patterns defined.

failed_re = re.compile(r'failed|rejected|echec|echoue|annule|canceled|cancelled', re.IGNORECASE)
airtime_re = re.compile(r'\bairtime\b', re.IGNORECASE)
balance_re = re.compile(r'\b(solde|balance)\b', re.IGNORECASE)
adjustment_re = re.compile(r'\badjustment|ajustement\b', re.IGNORECASE)
otp_re = re.compile(r'\b(code|otp|login|verifier|verification)\b', re.IGNORECASE)
promo_re = re.compile(r'\b(bonus|promo|promotion|cadeau|win|gagner|scholarship)\b', re.IGNORECASE)

print("Regex patterns defined for:")
print("- Failed transactions")
print("- Airtime purchases")
print("- Balance inquiries")
print("- Account adjustments")
print("- OTP/security codes")
print("- Promotional messages")

Regex patterns defined for:
- Failed transactions
- Airtime purchases
- Balance inquiries
- Account adjustments
- OTP/security codes
- Promotional messages


In [17]:
# Cell 5: Tag transaction status and types
# This cell applies the regex patterns to categorize messages and standardize transaction types.
# Expected output: Preview of tagged data.

raw['status'] = raw['content'].apply(lambda text: 'failed' if failed_re.search(str(text)) else 'success')

raw['transaction_type'] = raw['transaction_type'].astype(str).str.lower()
raw.loc[raw['content'].str.contains(airtime_re, na=False), 'transaction_type'] = 'airtime'
raw.loc[raw['content'].str.contains(balance_re, na=False), 'transaction_type'] = 'balance'
raw.loc[raw['content'].str.contains(adjustment_re, na=False), 'transaction_type'] = 'adjustment'

print("Status and transaction types tagged.")
print(f"Status distribution: {raw['status'].value_counts().to_dict()}")
print(f"Transaction type distribution: {raw['transaction_type'].value_counts().head().to_dict()}")

display(raw[['content', 'status', 'transaction_type']].head())

Status and transaction types tagged.
Status distribution: {'success': 25899, 'failed': 97}
Transaction type distribution: {'other': 21227, 'balance': 4412, 'receive': 227, 'airtime': 86, 'payment': 34}


C:\Users\USER PRO\AppData\Local\Temp\ipykernel_22188\4141115201.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  raw.loc[raw['content'].str.contains(balance_re, na=False), 'transaction_type'] = 'balance'


,content,status,transaction_type
0,Mr. X: 5032 FCFA ; Solde disponible: 5032 FCFA...,success,balance
1,Transfert de 2500 FCFA effectue avec succes a ...,success,balance
2,Mr. X: 1977 FCFA ; Solde disponible: 1977 FCFA...,success,balance
3,Une transaction de 1000 XAF effectue par MTNC ...,success,balance
4,Une transaction de 250 XAF effectue par MTNC B...,success,balance


In [18]:
# Cell 6: Filter out non-behavioral messages
# This cell removes OTP and promotional messages that don't represent user activity.
# Expected output: Number of messages removed.

otp_mask = raw['content'].str.contains(otp_re, na=False)
promo_mask = raw['content'].str.contains(promo_re, na=False)
cleaned = raw[~otp_mask & ~promo_mask].copy()

print(f'Removed {otp_mask.sum() + promo_mask.sum():,} OTP/promotional rows.')
print(f'Remaining rows: {len(cleaned):,}')

display(cleaned.head())

C:\Users\USER PRO\AppData\Local\Temp\ipykernel_22188\482768637.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  otp_mask = raw['content'].str.contains(otp_re, na=False)
C:\Users\USER PRO\AppData\Local\Temp\ipykernel_22188\482768637.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  promo_mask = raw['content'].str.contains(promo_re, na=False)


Removed 2,565 OTP/promotional rows.
Remaining rows: 23,431


,source_file,user_id,date,time,datetime,direction,contact,phone,type,transaction_type,amount,currency,content,transaction_id,status
0,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-05-31,12:00:21,2024-05-31 12:00:21,Received,MobileMoney,MobileMoney,SMS,balance,5032.0,XAF,Mr. X: 5032 FCFA ; Solde disponible: 5032 FCFA...,txn-5c2755d7-c05c-5128-ab39-c12c662521df,success
2,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-02,18:45:25,2024-06-02 18:45:25,Received,MobileMoney,MobileMoney,SMS,balance,1977.0,XAF,Mr. X: 1977 FCFA ; Solde disponible: 1977 FCFA...,txn-2b976feb-88d1-5db3-bf43-8d37f17cf58e,success
5,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-04,19:56:11,2024-06-04 19:56:11,Received,MobileMoney,MobileMoney,SMS,balance,727.0,XAF,Mr. X: 727 FCFA ; Solde disponible: 727 FCFA ;...,txn-a0930702-a2c8-574a-a7a6-bd7f345f162c,success
10,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-07,10:44:12,2024-06-07 10:44:12,Received,MobileMoney,MobileMoney,SMS,balance,10427.0,XAF,Mr. X: 10427 FCFA ; Solde disponible: 10427 FC...,txn-f847bdd9-7623-5e25-955d-1b57cfa9f019,success
12,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-07,15:02:56,2024-06-07 15:02:56,Received,MobileMoney,MobileMoney,SMS,adjustment,100.0,XAF,An adjustment has been made and 100 XAF has be...,txn-6ec6356c-85ee-5ea8-984c-c6b90d59c3dd,success


In [19]:
# Cell 7: Remove duplicates and invalid records
# This cell drops exact duplicates and rows with missing critical fields.
# Expected output: Number of rows after cleaning.

cleaned = cleaned.dropna(subset=['datetime', 'content'])
cleaned = cleaned.drop_duplicates(subset=['user_id', 'datetime', 'content', 'amount', 'transaction_type'])

print(f'After removing duplicates and missing data: {len(cleaned):,} rows')

After removing duplicates and missing data: 2,650 rows


In [20]:
# Cell 8: Filter to relevant transaction types
# This cell keeps only transaction categories that represent user activity.
# Expected output: Final transaction type distribution.

keep_types = {'receive', 'withdraw', 'transfer', 'payment', 'airtime', 'balance', 'adjustment'}
cleaned = cleaned[cleaned['transaction_type'].isin(keep_types)].copy()

print(f'Kept transaction types: {sorted(keep_types)}')
print(f'Final transaction type distribution:\n{cleaned["transaction_type"].value_counts()}')

Kept transaction types: ['adjustment', 'airtime', 'balance', 'payment', 'receive', 'transfer', 'withdraw']
Final transaction type distribution:
transaction_type
balance       2144
receive        224
airtime         71
payment         34
adjustment       7
transfer         3
Name: count, dtype: int64


In [21]:
# Cell 9: Handle amount fields
# This cell fills missing amounts for balance messages and ensures numeric conversion.
# Expected output: Summary of amount processing.

cleaned.loc[(cleaned['transaction_type'] == 'balance') & (cleaned['amount'].isna()), 'amount'] = 0
cleaned['amount'] = pd.to_numeric(cleaned['amount'], errors='coerce')
cleaned = cleaned[cleaned['amount'].fillna(0) >= 0]
cleaned = cleaned[~cleaned['amount'].isna() | (cleaned['transaction_type'] == 'balance')]

print(f'Final dataset: {len(cleaned):,} rows')
print(f'Amount range: {cleaned["amount"].min():.2f} to {cleaned["amount"].max():.2f}')

Final dataset: 2,468 rows
Amount range: 0.00 to 237681952519.00


In [22]:
# Cell 10: Save cleaned data and quality report
# This cell saves the final cleaned dataset and generates a missing value report.
# Expected output: Confirmation of saved files and quality summary.

cleaned.to_csv(cleaned_path, index=False)
quality = cleaned.isna().mean().reset_index()
quality.columns = ['column', 'missing_rate']
quality.to_csv(quality_path, index=False)

print(f'Cleaned dataset saved to: {cleaned_path}')
print(f'Quality report saved to: {quality_path}')
print(f'Cleaned dataset contains {len(cleaned):,} rows.')

display(cleaned.head())
display(quality.head())

Cleaned dataset saved to: d:\one drivee\OneDrive\Desktop\Final data sc\project\submission\DataScience_Final_GroupA_TeamGamma\2_Data_Cleaning\cleaned_data.csv
Quality report saved to: d:\one drivee\OneDrive\Desktop\Final data sc\project\submission\DataScience_Final_GroupA_TeamGamma\2_Data_Cleaning\data_quality_report.csv
Cleaned dataset contains 2,468 rows.


,source_file,user_id,date,time,datetime,direction,contact,phone,type,transaction_type,amount,currency,content,transaction_id,status
0,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-05-31,12:00:21,2024-05-31 12:00:21,Received,MobileMoney,MobileMoney,SMS,balance,5032.0,XAF,Mr. X: 5032 FCFA ; Solde disponible: 5032 FCFA...,txn-5c2755d7-c05c-5128-ab39-c12c662521df,success
2,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-02,18:45:25,2024-06-02 18:45:25,Received,MobileMoney,MobileMoney,SMS,balance,1977.0,XAF,Mr. X: 1977 FCFA ; Solde disponible: 1977 FCFA...,txn-2b976feb-88d1-5db3-bf43-8d37f17cf58e,success
5,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-04,19:56:11,2024-06-04 19:56:11,Received,MobileMoney,MobileMoney,SMS,balance,727.0,XAF,Mr. X: 727 FCFA ; Solde disponible: 727 FCFA ;...,txn-a0930702-a2c8-574a-a7a6-bd7f345f162c,success
10,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-07,10:44:12,2024-06-07 10:44:12,Received,MobileMoney,MobileMoney,SMS,balance,10427.0,XAF,Mr. X: 10427 FCFA ; Solde disponible: 10427 FC...,txn-f847bdd9-7623-5e25-955d-1b57cfa9f019,success
12,Messages with MobileMoney 2026-03-20 190409.csv,msg_2026-03-20_190409,2024-06-07,15:02:56,2024-06-07 15:02:56,Received,MobileMoney,MobileMoney,SMS,adjustment,100.0,XAF,An adjustment has been made and 100 XAF has be...,txn-6ec6356c-85ee-5ea8-984c-c6b90d59c3dd,success


,column,missing_rate
0,source_file,0.0
1,user_id,0.0
2,date,0.0
3,time,0.0
4,datetime,0.0
